# **Code**

In [2]:
import boto3, os
import pandas as pd
import pyarrow.parquet as pq

from io import StringIO, BytesIO
from dotenv import load_dotenv


In [3]:
load_dotenv()

True

In [4]:
s3 = boto3.client(
    's3',
    region_name=os.getenv('AWS_REGION'),
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
)

In [5]:
response = s3.get_object(Bucket=os.getenv('S3_BUCKET_NAME'), Key='raw/catalog/products.csv')
df_products = pd.read_csv(StringIO(response['Body'].read().decode('utf-8')))

print(df_products.shape)
print(df_products.dtypes)
print(df_products.head())
print(df_products.describe())

(228, 21)
product_id                  str
product_uuid                str
brand                       str
model_name                  str
colorway                    str
category                    str
subcategory                 str
slug                        str
display_name                str
price_usd               float64
_internal_cost_usd      float64
_supplier_id                str
_warehouse_location         str
_internal_cost_code         str
available_sizes_json        str
image_urls                  str
weight_grams              int64
is_active                  bool
is_hype_product            bool
created_at                  str
tags                        str
dtype: object
  product_id                          product_uuid   brand  \
0   KE-10023  d5fedc1b-03b3-42b9-8d5e-48a3469b6434    Nike   
1   KE-10093  c4e7be0d-bc56-43ff-b369-e2a6ea91dfc5  Adidas   
2   KE-10218  b7ef4692-68cf-44a9-95f4-2bc9231580cf  Adidas   
3   KE-10203  4bea0a64-e959-441b-9773-ab6d258162df    Pu

In [6]:
response = s3.get_object(Bucket=os.getenv('S3_BUCKET_NAME'), Key='raw/reviews/reviews.jsonl')
jsonl_content = response['Body'].read().decode('utf-8')

df_reviews = pd.read_json(StringIO(jsonl_content), lines=True)

print(df_reviews.shape)
print(df_reviews.dtypes)
print(df_reviews.head())

(2930, 20)
review_id                             str
product_id                            str
product_name                          str
user_id                               str
user_name                             str
rating                              int64
title                                 str
body                                  str
verified_purchase                    bool
submitted_at          datetime64[us, UTC]
moderation_status                     str
moderated_at          datetime64[us, UTC]
helpful_votes                       int64
reported                             bool
photos_count                        int64
_moderation_score                 float64
_sentiment_raw                    float64
_toxicity_score                   float64
_language_detected                    str
_review_source                        str
dtype: object
                              review_id product_id  \
0  16336ca4-ccfd-4089-999d-19cb0960f2a7   KE-10062   
1  46cfa7c2-7a8e-491f-b15f-

In [7]:
response = s3.get_object(
    Bucket=os.getenv('S3_BUCKET_NAME'),
    Key='raw/clickstream/dt=2026-02-05/part-00001.snappy.parquet'
)

table = pq.read_table(BytesIO(response['Body'].read()))
df_click = table.to_pandas()

print(df_click.shape)
print(df_click.dtypes)
print(df_click.head())



(5000, 27)
event_id                   str
event_type                 str
timestamp                  str
timestamp_epoch_ms       int64
user_id                    str
session_id                 str
page_url                   str
page_path                  str
page_type                  str
referrer_url               str
referrer_source            str
user_agent_raw             str
ip_address                 str
viewport_width           int64
viewport_height          int64
screen_color_depth       int64
device_pixel_ratio     float64
accept_language            str
is_bot                    bool
_ga_client_id              str
_gtm_container_id          str
_dom_interactive_ms      int64
_dom_complete_ms         int64
_ttfb_ms                 int64
_connection_type           str
_js_heap_size_mb       float64
_consent_string            str
dtype: object
                               event_id event_type  \
0  ac0e758c-cc0b-4f45-b1c7-2f63dc1a0af1   pageview   
1  7d534b62-0fae-493c-897b-3dd

# **Questions**

In [8]:
df_products.columns

Index(['product_id', 'product_uuid', 'brand', 'model_name', 'colorway',
       'category', 'subcategory', 'slug', 'display_name', 'price_usd',
       '_internal_cost_usd', '_supplier_id', '_warehouse_location',
       '_internal_cost_code', 'available_sizes_json', 'image_urls',
       'weight_grams', 'is_active', 'is_hype_product', 'created_at', 'tags'],
      dtype='str')

In [ ]:
users_response = s3.get_object(
    Bucket=os.getenv('S3_BUCKET_NAME'),
    Key='raw/users/users.csv'
)

df_users = pd.read_csv(StringIO(users_response['Body'].read().decode('utf-8')))

In [10]:
df_users.columns

Index(['user_id', 'user_uuid', 'email', 'first_name', 'last_name', 'phone',
       'gender', 'date_of_birth', 'country_code', 'country_name', 'city',
       'address_line_1', 'address_line_2', 'postal_code', 'timezone',
       'registration_date', 'is_verified', 'loyalty_tier', 'newsletter_opt_in',
       'preferred_language', '_hashed_password', '_ga_client_id', '_fbp',
       '_device_fingerprint', '_last_ip', '_failed_login_count',
       '_account_flags', '_internal_segment_id'],
      dtype='str')

In [11]:
df_users[['_hashed_password', '_last_ip']]

,_hashed_password,_last_ip
0,$2b$12$4cdd64ea06c59f9f0a93434701564a33437050c...,98.116.213.62
1,$2b$12$89ed6c642bf0d24f3f3f4bd1324c5e8a29d8738...,76.211.210.78
2,$2b$12$30b2989c477c6ccbdd4491479c62dce4ce4b9b2...,64.83.98.58
3,$2b$12$be0342d0b45a010fee5b58b8b697cd4a1230c40...,108.233.21.225
4,$2b$12$82ce47b23e77e9c53e777172aacbccf2576c914...,76.230.12.46
...,...,...
4995,$2b$12$fe52c57cc1fc4cbb2686c69828b2ca6bec112b7...,208.247.13.36
4996,$2b$12$c1afcf6f71e9801c2cb98ec34b1d389d0331205...,47.198.27.27
4997,$2b$12$172977bc2836fab5a8348d42bd59aeab24bf9c2...,50.94.91.191
4998,$2b$12$9f6c882c82d84a9dfd35df84a99e00e45becce8...,76.65.144.138


In [12]:
orders_response = s3.get_object(
    Bucket=os.getenv('S3_BUCKET_NAME'),
    Key='raw/orders/orders.csv'
)

df_orders = pd.read_csv(StringIO(orders_response['Body'].read().decode('utf-8')))

In [13]:
df_orders['status'].unique()

<ArrowStringArray>
['delivered', 'shipped', 'returned', 'chargeback', 'cancelled', 'processing']
Length: 6, dtype: str

In [14]:
orders_lines_response = s3.get_object(
    Bucket=os.getenv('S3_BUCKET_NAME'),
    Key='raw/order_line_items/order_line_items.csv'
)

df_orders_lines = pd.read_csv(StringIO(orders_lines_response['Body'].read().decode('utf-8')))

In [15]:
df_orders_lines.columns

Index(['line_item_id', 'order_id', 'product_id', 'product_name', 'brand',
       'category', 'sku', 'selected_size', 'colorway', 'quantity',
       'unit_price_usd', 'line_total_usd', 'weight_grams', '_warehouse_id',
       '_internal_batch_code', '_pick_slot'],
      dtype='str')

In [16]:
df_orders_lines['check'] = df_orders_lines['line_total_usd'] == (df_orders_lines['unit_price_usd'] * df_orders_lines['quantity'])

In [17]:
print(f"Number of rows with incorrect line totals: {df_orders_lines[df_orders_lines['check'] == False].shape[0]}")
print(f"Number of rows with correct line totals: {df_orders_lines[df_orders_lines['check'] == True].shape[0]}")

Number of rows with incorrect line totals: 430
Number of rows with correct line totals: 30454


In [18]:
df_orders_lines[df_orders_lines['check'] == False][['line_total_usd', 'unit_price_usd', 'quantity', 'check']]

,line_total_usd,unit_price_usd,quantity,check
48,400.44,133.48,3,False
119,466.92,155.64,3,False
206,373.20,124.40,3,False
231,374.16,124.72,3,False
246,215.04,71.68,3,False
...,...,...,...,...
30245,437.67,145.89,3,False
30353,166.14,55.38,3,False
30567,437.67,145.89,3,False
30679,338.16,112.72,3,False


In [19]:
df_reviews

,review_id,product_id,product_name,user_id,user_name,rating,title,body,verified_purchase,submitted_at,moderation_status,moderated_at,helpful_votes,reported,photos_count,_moderation_score,_sentiment_raw,_toxicity_score,_language_detected,_review_source
0,16336ca4-ccfd-4089-999d-19cb0960f2a7,KE-10062,Nike ACG Dri-FIT Tee Georgetown,USR-011515,Chloe H.,5,Must have!,"These are insane, the quality is top notch. Fi...",True,2026-02-05 20:21:12.721267+00:00,pending,NaT,0,False,0,0.3321,0.8989,0.1277,en,email_followup
1,46cfa7c2-7a8e-491f-b15f-91c52e936e42,KE-10141,Jordan Air Jordan 11 Retro Court Purple,USR-010333,Zoey C.,4,Nice but runs big,"Great sneakers, comfortable right out of the b...",True,2026-02-05 16:48:11.819267+00:00,approved,2026-02-07 03:48:11.819267+00:00,0,False,0,0.2997,0.7068,0.2001,en,mobile_app
2,804cb048-dafd-4265-84be-50499a7ad9f8,KE-10160,New Balance 574 Lucky Green,USR-013884,Luke H.,4,Good value for money,"Nice quality, true to photos. Shipping took a ...",False,2026-02-05 16:39:44.101262+00:00,rejected,2026-02-07 07:39:44.101262+00:00,2,False,0,0.4848,0.6897,0.0802,es,web
3,483504a3-ea06-4b77-9bb4-b3ac0b60b2d6,KE-10122,Adidas Adicolor Classics Tee Vintage Green,USR-014915,Abigail K.,5,Incredible quality,Best purchase of the year. I was torn between ...,False,2026-02-05 12:10:28.985580+00:00,approved,2026-02-05 19:10:28.985580+00:00,2,False,0,0.1966,0.9925,0.0226,es,web
4,359d1fc4-83f4-4b9a-88ce-603da1095ea0,KE-10216,Nike Heritage86 Futura Cap,USR-013935,Nora C.,4,"Great overall, minor flaw","Nice quality, true to photos. Shipping took a ...",True,2026-02-05 18:04:28.878323+00:00,approved,2026-02-06 02:04:28.878323+00:00,0,False,2,0.8811,0.6270,0.0925,en,web
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,1fb00109-a227-4504-b8f1-16de7fe26308,KE-10106,Adidas Ultraboost 24 Navy/White,USR-010366,Chloe T.,5,Best purchase ever,"These are insane, the quality is top notch. Fi...",True,2026-03-06 14:51:57.254936+00:00,approved,2026-03-06 15:51:57.254936+00:00,5,False,1,0.9794,1.0418,0.1497,en,mobile_app
2926,9303cdd9-5df9-49ed-9ce0-d6b7bed3db8e,KE-10105,Adidas NMD_R1 Washed Denim,USR-011925,Cole S.,5,Exactly as pictured,"I love these shoes, they're incredibly comfort...",True,2026-03-06 19:27:11.586423+00:00,approved,2026-03-07 16:27:11.586423+00:00,10,False,3,0.2283,1.0156,0.2465,en,mobile_app
2927,f8300b89-8fed-4d8d-bae1-ff91170d295b,KE-10129,Jordan Air Jordan 1 Low SE White/Black,USR-012325,Caleb G.,4,Solid pair,"Nice quality, true to photos. Shipping took a ...",True,2026-03-06 11:26:29.730106+00:00,approved,2026-03-07 19:26:29.730106+00:00,5,False,0,0.3025,0.8240,0.0784,en,web
2928,815d8e4a-0e8e-4a9f-b2cf-e53183a69459,KE-10212,Nike Heritage Waist Bag,USR-012403,Addison A.,5,Love them,"These are insane, the quality is top notch. Fi...",True,2026-03-06 21:20:20.877993+00:00,approved,2026-03-07 04:20:20.877993+00:00,0,False,2,0.6872,0.9422,0.1281,en,web


In [35]:
df_click['event_type'].unique()

<ArrowStringArray>
['pageview']
Length: 1, dtype: str

In [29]:
df_click.columns

Index(['event_id', 'event_type', 'timestamp', 'timestamp_epoch_ms', 'user_id',
       'session_id', 'page_url', 'page_path', 'page_type', 'referrer_url',
       'referrer_source', 'user_agent_raw', 'ip_address', 'viewport_width',
       'viewport_height', 'screen_color_depth', 'device_pixel_ratio',
       'accept_language', 'is_bot', '_ga_client_id', '_gtm_container_id',
       '_dom_interactive_ms', '_dom_complete_ms', '_ttfb_ms',
       '_connection_type', '_js_heap_size_mb', '_consent_string'],
      dtype='str')